# Agent for S05E03: Functional Development

This notebook implements an agent that can develop and extend its own functionality, as discussed in the S05E03 lesson about evolving AI applications.

## Features:
1. Dynamic tool registration
2. Self-improvement through reflection
3. Code generation and execution
4. Learning from interactions

In [ ]:
# Install required packages
!pip install openai langchain

In [ ]:
from openai import OpenAI
from langchain.tools import tool
import json
import inspect

class SelfDevelopingAgent:
    def __init__(self, api_key):
        self.client = OpenAI(api_key=api_key)
        self.tools = {}
        self.experience = []
        self.register_base_tools()
    
    def register_base_tools(self):
        """Register initial tools"""
        @tool
        def add_numbers(a: int, b: int) -> int:
            """Add two numbers"""
            return a + b
        
        @tool
        def multiply_numbers(a: int, b: int) -> int:
            """Multiply two numbers"""
            return a * b
        
        self.tools['add_numbers'] = add_numbers
        self.tools['multiply_numbers'] = multiply_numbers
    
    def generate_new_tool(self, description):
        """Generate a new tool based on description"""
        prompt = f"""
        Create a Python function tool based on this description: {description}
        
        Return a JSON with:
        - name: function name
        - code: the function code as string
        - description: tool description
        """
        
        response = self.client.chat.completions.create(
            model="gpt-4",
            messages=[{"role": "user", "content": prompt}]
        )
        
        try:
            result = json.loads(response.choices[0].message.content)
            return result
        except:
            return None
    
    def register_tool(self, tool_data):
        """Register a new tool dynamically"""
        try:
            # Execute the code to define the function
            exec(tool_data['code'], globals())
            func = globals()[tool_data['name']]
            
            # Create tool decorator
            tool_func = tool(func)
            tool_func.description = tool_data['description']
            
            self.tools[tool_data['name']] = tool_func
            return True
        except Exception as e:
            print(f"Error registering tool: {e}")
            return False
    
    def reflect_and_improve(self):
        """Analyze experience and suggest improvements"""
        if not self.experience:
            return "No experience to analyze"
        
        prompt = f"""
        Analyze this agent experience and suggest improvements:
        {self.experience[-5:]}  # Last 5 interactions
        
        Suggest:
        1. New tools to add
        2. Improvements to existing functionality
        3. Better error handling
        """
        
        response = self.client.chat.completions.create(
            model="gpt-4",
            messages=[{"role": "user", "content": prompt}]
        )
        
        return response.choices[0].message.content
    
    def execute_task(self, task):
        """Execute a task using available tools"""
        available_tools = list(self.tools.keys())
        
        prompt = f"""
        Available tools: {available_tools}
        Task: {task}
        
        Decide which tool to use or if you need to create a new one.
        Return JSON with 'action': 'use_tool' or 'create_tool', and 'tool_name' or 'tool_description'
        """
        
        response = self.client.chat.completions.create(
            model="gpt-4",
            messages=[{"role": "user", "content": prompt}]
        )
        
        try:
            decision = json.loads(response.choices[0].message.content)
            
            if decision['action'] == 'create_tool':
                tool_data = self.generate_new_tool(decision['tool_description'])
                if tool_data and self.register_tool(tool_data):
                    result = f"Created and registered new tool: {tool_data['name']}"
                else:
                    result = "Failed to create tool"
            else:
                tool_name = decision['tool_name']
                if tool_name in self.tools:
                    # For simplicity, assume tool execution
                    result = f"Executed tool: {tool_name}"
                else:
                    result = f"Tool {tool_name} not found"
            
            self.experience.append(f"Task: {task}, Result: {result}")
            return result
            
        except Exception as e:
            return f"Error: {str(e)}"

# Initialize agent (requires OpenAI API key)
# agent = SelfDevelopingAgent(api_key='your-openai-key')

In [ ]:
# Example usage (uncomment and add API key)
# print("Available tools:", list(agent.tools.keys()))
# result = agent.execute_task("Calculate 15 divided by 3")
# print(result)
# improvement = agent.reflect_and_improve()
# print("Improvement suggestions:", improvement)